# 02 · Regresión logística: clase de utilidad de una parcela

**Responsable:** Fabiana Leonardo (@fabileoruf). **Historia:** [#38](https://github.com/IA-P1-BCN/g3-Clasificacion-Multiclase/issues/38); subtareas #52, #53 y #54.

Predecimos `Utility` en las cinco clases `none`, `low`, `average`, `good` y `best`.
El objetivo es superar el Dummy con un pipeline reproducible y un gap de F1 macro inferior a 5 puntos.
La regresión logística calcula probabilidades multiclase; no interpreta la distancia entre clases como una regresión numérica.

Se usan los CSV compartidos y la biblioteca obligatoria `comun.py`, según [los acuerdos](docs/acuerdos.md).
No se vuelve a hacer el split ni la limpieza del EDA. El conjunto reservado para la evaluación final se mantiene sin leer.

**Ejecución:** abrir este notebook desde la raíz del repositorio, seleccionar el entorno `.venv` con las dependencias de `requirements.txt` y ejecutar todas las celdas, en orden.


## 1. Entorno y funciones comunes

Las decisiones específicas del algoritmo están aquí; carga, baseline, CV, regla `none`, evaluación y exportación se importan de `comun.py`.


In [ ]:
import json
import subprocess
import sys
from functools import partial
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from comun import (
    CLASES, GAP_MAXIMO, NUMERICAS, RAIZ, SEMILLA,
    baseline_dummy, cargar_datos, conclusion_modelo, cv_comun,
    elegir_sin_sobreajuste, errores_frecuentes, evaluar_val,
    f1_por_fold, guardar_modelo, mostrar_evaluacion,
    separar_xy, sin_medidas,
)

NOMBRE_RL = "02_regresion_logistica"
print("Python:", sys.version.split()[0])
print({paquete: version(paquete) for paquete in ["numpy", "pandas", "scikit-learn", "joblib"]})


## 2. Datos compartidos

La limpieza de `01_eda` ya excluye las parcelas sin medidas de train. Validación las conserva.
`Abbrev` se conserva también para la comprobación de generalización a sitios nuevos.
Las distribuciones que se muestran aquí corresponden únicamente a entrenamiento.


In [ ]:
train_rl, val_rl = cargar_datos()
X_train_rl, y_train_rl = separar_xy(train_rl)
X_val_rl, y_val_rl = separar_xy(val_rl)

assert set(y_train_rl.unique()) == set(CLASES)
assert not sin_medidas(X_train_rl).any(), "Revisar los CSV comunes: train contiene parcelas sin medidas."
print(f"Train: {len(train_rl)} parcelas. Validación: {len(val_rl)} parcelas.")
print(f"Validación sin medidas: {sin_medidas(X_val_rl).sum()} (se aplica la regla none).")
display(y_train_rl.value_counts().reindex(CLASES).rename_axis("clase de utilidad").to_frame("parcelas de entrenamiento"))


## 3. Preprocesado y decisiones

| Decisión | Por qué |
|---|---|
| Usar las 8 medidas numéricas de `NUMERICAS` | Son mediciones de campo; deben estar disponibles cuando se prediga. |
| Imputar con la mediana y aplicar `StandardScaler` | Resolver los ausentes de supervivencia y diámetro; evitar que las unidades dominen la regularización. |
| Codificar `Abbrev` y `Sp` con one-hot | Son categorías sin orden numérico. Las categorías desconocidas se toleran; los valores raros se agrupan si aparecen menos de 3 veces en el fold. |
| Excluir `Rep` y `PMCno` mediante `remainder="drop"` | Son identificadores del diseño del ensayo y de procedencia; no se supone una relación ordinal entre sus números y la utilidad. |
| Conservar `DBH`/`Ht` y las tres medidas de forma, con regularización L2 | El EDA detectó correlaciones. L2 limita los coeficientes sin eliminar arbitrariamente una medida; dificulta atribuir importancia individual a variables correlacionadas. |
| Comparar `class_weight=None` y `"balanced"` | Atender al desbalance moderado sin crear parcelas sintéticas. La elección se hace solo con CV sobre train. |

Todo el preprocesado está dentro del `Pipeline`: cada fold ajusta sus propias medianas, escalas y categorías.
Las columnas descartadas siguen en el DataFrame de entrada para que la biblioteca común pueda comprobar los sitios y las medidas.

Referencias de la API: [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) y [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).


In [ ]:
numericas_rl = list(NUMERICAS)
categoricas_rl = ["Abbrev", "Sp"]

preprocesado_rl = ColumnTransformer(
    transformers=[
        ("numericas", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("escalar", StandardScaler()),
        ]), numericas_rl),
        ("categoricas", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("codificar", OneHotEncoder(handle_unknown="ignore", min_frequency=3)),
        ]), categoricas_rl),
    ],
    remainder="drop",
)

pipeline_rl = Pipeline([
    ("preprocesado", preprocesado_rl),
    ("logistica", LogisticRegression(
        solver="lbfgs", l1_ratio=0.0, max_iter=5000, random_state=SEMILLA,
    )),
])


## 4. Baseline común

El Dummy predice la clase más frecuente de entrenamiento. La regla `none` se aplica también al baseline, para comparar en las mismas condiciones.


In [ ]:
dummy_rl = baseline_dummy(X_train_rl, y_train_rl, X_val_rl, y_val_rl)


## 5. Selección de hiperparámetros, solo con entrenamiento

La rejilla se fija antes de evaluar el modelo en validación. Un `C` pequeño implica mayor regularización.
Se usa `elegir_sin_sobreajuste`: mayor F1 macro en CV entre candidatos con gap train–CV inferior a 5 puntos; en empate se favorece menor `C`.
Si ninguno cumple, la biblioteca elige el menor gap y lo avisa. No se reajusta la rejilla mirando validación.


In [ ]:
rejilla_rl = {
    "logistica__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0],
    "logistica__class_weight": [None, "balanced"],
}
busqueda_rl = GridSearchCV(
    pipeline_rl,
    param_grid=rejilla_rl,
    scoring="f1_macro",
    cv=cv_comun(),
    return_train_score=True,
    refit=partial(elegir_sin_sobreajuste, desempate=[("param_logistica__C", True)]),
    n_jobs=-1,
    error_score="raise",
)
busqueda_rl.fit(X_train_rl, y_train_rl)
modelo_rl = busqueda_rl.best_estimator_

tabla_busqueda_rl = pd.DataFrame(busqueda_rl.cv_results_)
tabla_busqueda_rl["gap_cv_puntos"] = 100 * (tabla_busqueda_rl["mean_train_score"] - tabla_busqueda_rl["mean_test_score"])
tabla_busqueda_rl["elegido"] = tabla_busqueda_rl.index == busqueda_rl.best_index_
display(tabla_busqueda_rl[[
    "param_logistica__C", "param_logistica__class_weight",
    "mean_train_score", "mean_test_score", "std_test_score", "gap_cv_puntos", "elegido",
]].sort_values("mean_test_score", ascending=False).round(4))
print("Hiperparámetros elegidos solo con train:", busqueda_rl.best_params_)
print("Iteraciones hasta converger:", modelo_rl.named_steps["logistica"].n_iter_)


## 6. Validación cruzada común y por sitio — #52

Se presentan los cinco folds, la media y la desviación con ambos esquemas comunes.
La comprobación por sitio utiliza `Abbrev` como grupo y no se utiliza para retocar el modelo.
La CV común comparte los folds usados para elegir hiperparámetros: su resultado es descriptivo tras selección, no una estimación independiente de ella.


In [ ]:
cv_rl = f1_por_fold(modelo_rl, X_train_rl, y_train_rl)
print(f"Diferencia CV común − CV por sitio: {100 * (cv_rl['f1'] - cv_rl['f1_sitio']):.2f} puntos.")


## 7. Evaluación en validación y overfitting — #53

`evaluar_val` usa `predecir_con_regla`: las parcelas sin medidas reciben `none` **sin entrar al modelo**.
Se muestran accuracy, precision/recall/F1 por clase y matrices de confusión, tanto sobre toda la validación como solo sobre parcelas con medidas.

El gap acordado es `(F1 train − F1 validación con medidas) × 100`, **con signo**.
El resultado sobre toda la validación incluye el acierto de la regla y se usa para comparar los modelos del equipo.


In [ ]:
evaluacion_rl = evaluar_val(modelo_rl, X_train_rl, y_train_rl, X_val_rl, y_val_rl, dummy_rl)
mostrar_evaluacion(evaluacion_rl)

if evaluacion_rl["gap"] < -GAP_MAXIMO:
    print("La validación supera al entrenamiento por más de 5 puntos. Cumple la regla de sobreajuste, pero este split pequeño puede ser más favorable; no demuestra generalización perfecta.")
elif not evaluacion_rl["cumple_gap"]:
    print("No cumple el límite de sobreajuste: se documenta el resultado y no se ajustan hiperparámetros usando esta validación. No es candidato al modelo ganador con la regla actual.")
else:
    print("Cumple el límite de sobreajuste acordado. La comprobación por sitio y los errores siguen siendo necesarios para valorar sus límites.")


## 8. Interpretación por coeficientes — #54

La importancia nativa se resume como la media del valor absoluto de los coeficientes de las cinco clases, por variable transformada.
Las numéricas están estandarizadas; las indicadoras one-hot están en escala 0/1. Es una lectura del modelo, no una medida causal ni una comparación exacta entre tipos de variable.
Las variables correlacionadas pueden repartirse los coeficientes. La tabla por clase conserva el signo y permite ver qué asociaciones favorecen cada clase.


In [ ]:
nombres_variables_rl = modelo_rl.named_steps["preprocesado"].get_feature_names_out()
coeficientes_rl = pd.DataFrame(
    modelo_rl.named_steps["logistica"].coef_.T,
    index=nombres_variables_rl,
    columns=modelo_rl.named_steps["logistica"].classes_,
).reindex(columns=CLASES)
importancia_rl = coeficientes_rl.abs().mean(axis=1).sort_values(ascending=False)
principales_rl = importancia_rl.head(15)

fig_rl, ax_rl = plt.subplots(figsize=(9, 6))
principales_rl.sort_values().plot.barh(ax=ax_rl, color="#246b8e")
ax_rl.set(title="Regresión logística: 15 coeficientes de mayor magnitud media", xlabel="Media de |coeficiente| entre clases", ylabel="Variable transformada")
fig_rl.tight_layout()
plt.show()
display(coeficientes_rl.loc[principales_rl.index].round(3))


## 9. Análisis de errores — #54

Se ordenan las confusiones reales y se inspeccionan parcelas mal clasificadas junto con su vigor y otras medidas.
La escala de utilidad es ordenada, aunque la regresión logística multinomial no impone ese orden.
Una confusión entre clases vecinas puede ser compatible con medidas de campo parecidas; una de dos o más niveles requiere especial atención.
Estas explicaciones son hipótesis, no causas demostradas por el modelo.


In [ ]:
errores_rl = errores_frecuentes(evaluacion_rl["y"], evaluacion_rl["pred"])
display(errores_rl)

parcelas_error_rl = X_val_rl.assign(real=y_val_rl, predicha=evaluacion_rl["pred"])
parcelas_error_rl = parcelas_error_rl.loc[parcelas_error_rl["real"] != parcelas_error_rl["predicha"]]
display(parcelas_error_rl[["Abbrev", "Sp", "Vig", "Surv", "Ins_res", "real", "predicha"]].head(12))

for error_rl in errores_rl.head(3).itertuples(index=False):
    print(f"{error_rl.real} → {error_rl.predicha}: {error_rl.parcelas} parcelas; distancia de {error_rl.distancia} niveles.")
if cv_rl["f1_sitio"] < cv_rl["f1"]:
    print("La caída al separar sitios es compatible con dependencia del contexto del sitio. El rendimiento en sitios nuevos debe comunicarse con esta limitación (ADR-0002).")
print("Vig y el resto de medidas deben estar disponibles en la parcela al predecir. Si faltan todas, responde la regla; no se interpreta como éxito de la regresión logística.")


## 10. Exportación y recarga en una sesión nueva — #54

Se guarda el pipeline completo con la función común. Además de su comprobación de recarga, se abre un proceso Python independiente y se verifica que produce exactamente las mismas predicciones en validación.
La regla de parcelas sin medidas vive en `comun.py`: la app debe llamar a `predecir_con_regla`, no al pipeline directamente sobre esas parcelas.


In [ ]:
ruta_modelo_rl = guardar_modelo(modelo_rl, NOMBRE_RL, X_val_rl)

codigo_recarga_rl = """
import json
import joblib
import pandas as pd
from comun import DATOS, MODELOS, predecir_con_regla, separar_xy

modelo = joblib.load(MODELOS / "02_regresion_logistica.joblib")
validacion = pd.read_csv(DATOS / "val.csv", encoding="utf-8")
X, _ = separar_xy(validacion)
predicciones = predecir_con_regla(modelo, X)
print(json.dumps(predicciones.tolist()))
"""
sesion_nueva_rl = subprocess.run(
    [sys.executable, "-c", codigo_recarga_rl],
    cwd=RAIZ, check=True, capture_output=True, text=True, encoding="utf-8",
)
predicciones_recargadas_rl = json.loads(sesion_nueva_rl.stdout)
assert predicciones_recargadas_rl == evaluacion_rl["pred"].tolist()
print(f"Sesión Python nueva: {len(predicciones_recargadas_rl)} predicciones idénticas.")


## 11. Conclusión y entrega común

La tabla de requisitos y el JSON se generan con `conclusion_modelo`. Se entregan este notebook ejecutado, el pipeline y `resultados/02_regresion_logistica.json` para la comparación de `06_conclusiones`.

**Requisitos cualitativos:** preprocesado dentro del pipeline (§3); importancia nativa con gráfico y coeficientes por clase (§8); confusiones y parcelas mal clasificadas (§9); recarga en proceso independiente (§10).
El equipo elegirá el modelo ganador entre sus cuatro algoritmos. Este notebook no realiza la evaluación final del equipo.


In [ ]:
ruta_resultados_rl = conclusion_modelo(
    NOMBRE_RL, cv_rl, evaluacion_rl, dummy_rl, busqueda_rl.best_params_,
)
print(f"F1 macro en toda la validación: {evaluacion_rl['filas']['toda la validación']['F1 macro']:.3f}")
print(f"F1 macro solo con medidas: {evaluacion_rl['filas']['solo con medidas']['F1 macro']:.3f}")
print(f"Gap de sobreajuste: {evaluacion_rl['gap']:.2f} puntos; cumple: {evaluacion_rl['cumple_gap']}.")
